In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import embeddings as emb
import inference as inf
import boto3
from pathlib import Path

## Load vector stores from S3

In [ ]:
# Load recursive split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_recursive = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="recursive")

In [ ]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_basic")

In [ ]:
# Load unstructured (basic) split embeddings for model "intfloat/multilingual-e5-large-instruct"
e5_large_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/multilingual-e5-large-instruct", splitter_type="unstruct_by_title")

In [ ]:
# Load recursive split embeddings for model "intfloat/e5-small-v2"
e5_small_recursive = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="recursive")

In [ ]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_basic = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_basic")

In [ ]:
# Load unstructured (basic) split embeddings for model "intfloat/e5-small-v2"
e5_small_unstruct_by_title = emb.load_embedding_vector_store(model="intfloat/e5-small-v2", splitter_type="unstruct_by_title")

## Get user queries from AWS S3 

In [ ]:
# Get user queries from S3
base_dir = Path.cwd().parent 
local_file_dir = base_dir / "temp" 
queries = emb.download_from_s3(s3_path="zoning-code-questions.csv", output_path=local_file_dir)

## Run Inference using AWS Converse API

In [ ]:
# Define dictionary for embedding models 
embeddings = {
    "e5_large_recursive" : e5_large_recursive, 
    "e5_large_unstruct_basic" : e5_large_unstruct_basic,
    "e5_large_unstruct_by_title" : e5_large_unstruct_by_title,  
    "e5_small_recursive" : e5_small_recursive,  
    "e5_small_unstruct_basic" : e5_small_unstruct_basic,
    "e5_small_unstruct_by_title" : e5_small_unstruct_by_title
}

# Define list for question types
question_types = ["comp1-aud1-data1", "comp2-aud1-data1", "comp3-aud1-data1", "comp1-aud2-data1", "comp2-aud2-data1", "comp3-aud2-data1"]

In [ ]:
# Create output folder
local_path = base_dir / "output"
local_path.mkdir(exist_ok=True)

### Meta Llama 3.2 1B Instruct

In [ ]:
# Define model inference profile name
llama_inf_profile_name = "US Meta Llama 3.2 1B Instruct"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "meta-llama" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.run_conversation(llama_inf_profile_name, q_type, emb_vs, path, n_iter=3)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/meta-llama/{emb_name}") 


### Mistral AI Pixtral Large 2502

In [ ]:
# Define model inference profile name
mistra_inf_profile_name = "US Mistral Pixtral Large 25.02"

for emb_name, emb_vs in embeddings.items():

    # Create output folder
    path = local_path / "mistral-ai-pixtral" / emb_name
    path.mkdir(parents=True, exist_ok=True)
    
    for q_type in question_types: 
        # Run inference for model using specified questions and embedding vector store
        print(f"Running conversation for '{q_type}' questions using {emb_name} embeddings.")
        inf.run_conversation(mistra_inf_profile_name, q_type, emb_vs, path, n_iter=2)

    # Move output to AWS S3
    emb.upload_to_s3(path, f"output/mistral-ai-pixtral/{emb_name}") 
        